## 4 -> Customer Metrics

### Reading Data

In [12]:
import pandas as pd
import numpy as np

cleaned_df = pd.read_csv('../data/processed/cleaned_data.csv')
cleaned_df['TotalAmount'] = cleaned_df['Quantity'] * cleaned_df['UnitPrice']

In [13]:
cleaned_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


### Calculating Customer Metrics

In [14]:
customer_metrics = cleaned_df.groupby(by='CustomerID', as_index=False)\
  .agg(
    TotalSpent = ('TotalAmount', 'sum'),
    OrderCount = ('InvoiceNo', 'nunique'),
    FirstInvoiceDate = ('InvoiceDate', 'min'),
    LastInvoiceDate =('InvoiceDate', 'max')
  )

In [15]:
customer_metrics['LastInvoiceDate'] = pd.to_datetime(customer_metrics['LastInvoiceDate'])
customer_metrics['FirstInvoiceDate'] = pd.to_datetime(customer_metrics['FirstInvoiceDate'])

customer_metrics['CustomerLifespan'] = (customer_metrics['LastInvoiceDate'] - customer_metrics['FirstInvoiceDate']).dt.days

In [16]:
#using the maxinvoice date from the table instead of current date since the data is old
max_invoice_date = customer_metrics['LastInvoiceDate'].max()

customer_metrics['Recency'] = (max_invoice_date - customer_metrics['LastInvoiceDate']).dt.days

In [17]:
customer_metrics['CustomerLifespan'] = np.where(customer_metrics['CustomerLifespan'] == 0, \
                                                1, customer_metrics['CustomerLifespan'])

customer_metrics['PurchaseFrequency'] = (customer_metrics['OrderCount'] / (customer_metrics['CustomerLifespan'] / 30)
).round(2)

### Exporting Data

In [19]:
customer_metrics.to_csv('../data/processed/customer_metrics.csv', index=False)